# NL-to-SQL Pipeline — Dissertation Walkthrough

Traces a single NLQ through every stage of the pipeline: schema encoding → few-shot prompting → generation → guardrails → post-processing → schema validation → execution-guided repair (ReAct).

Each section explains the **design decision**, **what the code actually does**, and **why alternative approaches were rejected**.

In [ ]:
import os, sys, shutil
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    repo_dir = Path("/content/NLtoSQL")
    if not (repo_dir / "nl2sql").exists():
        if repo_dir.exists():
            shutil.rmtree(repo_dir)
        os.system(f"git clone https://github.com/MacKenzieOBrian/NLtoSQL.git {repo_dir}")
    os.chdir(repo_dir)
    os.system("pip -q install -r requirements.txt")

sys.path.insert(0, str(Path.cwd()))
print("working directory:", Path.cwd())
print("in Colab:", IN_COLAB)

In [ ]:
# real imports — no torch, no DB connection needed for schema/prompt/postprocess stages
from nl2sql.core.prompting   import make_few_shot_messages, SYSTEM_INSTRUCTIONS
from nl2sql.core.postprocess import normalize_sql, first_select_only, guarded_postprocess, RANKING_HINT_RE
from nl2sql.core.validation  import parse_schema_text, validate_sql
from nl2sql.core.llm         import extract_first_select
from nl2sql.core.sql_guardrails import clean_candidate_with_reason, _normalize_spaced_keywords
from nl2sql.agent.agent_tools import schema_to_text

print("imports ok")

---
## 1. Schema Representation — `schema_to_text()`

**What it does.** `schema_to_text()` (→ `nl2sql/agent/agent_tools.py:52`) converts the structured schema dict (built live from `INFORMATION_SCHEMA` by `build_schema_summary()`) into a compact prompt string:

```
customers(customerNumber, customerName, country, creditLimit)
orders(orderNumber, customerNumber, status, orderDate)
Join hints: orders.customerNumber = customers.customerNumber; ...
```

**Why this format?**
- **Full DDL** (`CREATE TABLE ...`) carries data types, constraints, and defaults — irrelevant to SQL generation and wastes context tokens on 7B-parameter models with a 4096-token limit.
- **JSON schema** is structurally verbose and not in pretraining distribution for SQL tasks.
- **`table(col, ...)` format** is a compact convention used by DIN-SQL and DAIL-SQL; it fits 5 tables in ≈10 lines.

**Why FK join hints?** Small models (7–8B) struggle to identify join paths from column names alone. Explicit `t1.col = t2.col` hints reduce hallucinated joins (Lin et al. [2]).

**Design alternatives not taken:**
- Column data types: not needed — the model writes SQL based on column *names*, not types.
- Primary key annotations: column order (PK first) already signals this implicitly.

In [ ]:
# Fake schema dict — same structure as what build_schema_summary() returns at runtime
# (nl2sql/core/schema.py queries INFORMATION_SCHEMA to build this; we hardcode it here)
schema_cache = {
    "tables": [
        {"name": "customers",    "columns": [{"name": "customerNumber"}, {"name": "customerName"}, {"name": "country"}, {"name": "creditLimit"}]},
        {"name": "orders",       "columns": [{"name": "orderNumber"}, {"name": "customerNumber"}, {"name": "status"}, {"name": "orderDate"}]},
        {"name": "orderdetails", "columns": [{"name": "orderNumber"}, {"name": "productCode"}, {"name": "quantityOrdered"}, {"name": "priceEach"}]},
        {"name": "products",     "columns": [{"name": "productCode"}, {"name": "productName"}, {"name": "productLine"}, {"name": "MSRP"}]},
        {"name": "payments",     "columns": [{"name": "customerNumber"}, {"name": "checkNumber"}, {"name": "amount"}, {"name": "paymentDate"}]},
    ],
    "foreign_keys": [
        {"table": "orders",       "column": "customerNumber", "ref_table": "customers",  "ref_column": "customerNumber"},
        {"table": "payments",     "column": "customerNumber", "ref_table": "customers",  "ref_column": "customerNumber"},
        {"table": "orderdetails", "column": "orderNumber",    "ref_table": "orders",     "ref_column": "orderNumber"},
    ],
}

schema_text = schema_to_text(schema_cache)

# --- annotate the output so the format is self-explaining ---
print("schema_to_text() output (what the model sees in the prompt):")
print()
lines = schema_text.splitlines()
table_lines = [l for l in lines if not l.startswith("Join")]
join_lines  = [l for l in lines if l.startswith("Join")]

print("  TABLE LINES  (one per table; columns are position-ordered, PK first):")
for l in table_lines:
    print(f"    {l}")
print()
print("  FK JOIN HINTS  (injected to anchor join paths for small models):")
for l in join_lines:
    print(f"    {l}")
print()
print(f"  Total tokens (approx): {len(schema_text.split())} words")
print(f"  vs. full DDL would be ~5x longer with types, constraints, defaults")

---
## 2. Few-Shot In-Context Learning — `make_few_shot_messages()`

**What it does.** `make_few_shot_messages()` (→ `nl2sql/core/prompting.py:23`) assembles a chat message list:

```
[system]    SYSTEM_INSTRUCTIONS  (fixed rules)
[user]      Schema Details: ...  (schema context)
[user]      Example Question: <nlq_1>   ← only present when k > 0
[assistant] <sql_1>;
...repeated k times...
[user]      Natural Language Question: <target_nlq>
```

**Why chat format?** Chat models (Llama-3-Instruct, Qwen-2.5-Instruct) are RLHF-tuned to the `[system] / [user] / [assistant]` alternation. Their tokenizer's `apply_chat_template()` inserts model-specific special tokens; raw string prompting bypasses this and degrades generation quality.

**Why schema before exemplars?** Placing the schema at position 2 (right after the system prompt) ensures it is within the model's "attention primacy" window. Exemplars that follow are conditioned on the schema.

**Why k=3?** The key empirical finding of this dissertation: k=0 → EM ≈ 0.5% (model generates correct SQL but in the wrong style/format); k=3 → EM ≈ 30% (Llama) / 36% (Qwen). Exemplars teach *formatting* as much as *reasoning*.

**Why ICL over fine-tuning?** QLoRA fine-tuning at k=3 still underperforms Base+k=3 by 5.2pp (Llama) / 8.0pp (Qwen) EX. Fine-tuning may suppress the model's ability to adapt to new exemplars in-context (see statistical results, Section 9).

**Design alternatives not taken:**
- k=1 or k=5: k=1 gives insufficient formatting signal; k=5 approaches context limit on 7B models with 4096-token prompts.
- Random exemplar sampling: uses a deterministic seed per NLQ (`f"{seed}:{nlq}"`) for reproducibility across runs.

In [ ]:
NLQ = "Top 3 countries by number of orders."

EXEMPLARS = [
    {"nlq": "List all customer names in Spain.",
     "sql": "SELECT customerName FROM customers WHERE country = 'Spain';"},
    {"nlq": "How many products are in each product line?",
     "sql": "SELECT productLine, COUNT(*) FROM products GROUP BY productLine;"},
    {"nlq": "What is the total payment amount received?",
     "sql": "SELECT SUM(amount) FROM payments;"},
]

msgs_k0 = make_few_shot_messages(schema=schema_text, exemplars=[],        nlq=NLQ)
msgs_k3 = make_few_shot_messages(schema=schema_text, exemplars=EXEMPLARS, nlq=NLQ)

# --- Show the actual system prompt so the design decisions are visible ---
print("=" * 60)
print("SYSTEM_INSTRUCTIONS (the fixed system message every run sees):")
print("=" * 60)
print(SYSTEM_INSTRUCTIONS)

# --- Show the full k=3 message list ---
print("=" * 60)
print(f"k=3 prompt: {len(msgs_k3)} messages")
print("=" * 60)
for i, m in enumerate(msgs_k3):
    role = m['role'].upper()
    body = m['content']
    # truncate schema/exemplar bodies for readability
    if len(body) > 120:
        body = body[:120].rstrip() + "…"
    print(f"  [{i}] [{role:<10}]  {body}")

print()
print(f"k=0: {len(msgs_k0)} messages  (system + schema + target question)")
print(f"k=3: {len(msgs_k3)} messages  (+ 2 messages per exemplar = +6)")
print()
print("Each exemplar adds [user: Example Question] + [assistant: SQL];")
print("this teaches the model the expected output format without any gradient update.")

---
## 3. SQL Extraction — `extract_first_select()`

**What it does.** `extract_first_select()` (→ `nl2sql/core/llm.py:38`) takes raw model output and returns the first syntactically plausible `SELECT` statement. It:
1. Strips markdown fences (` ```sql `, ` ```json `, ` ``` `)
2. Scans for `SELECT` at a line boundary (regex `SQL_START_RE`)
3. Terminates at the first `;`
4. **Rejects prose-FROM** — if the token after `FROM` is a stopword (`the`, `this`, `a`, ...), the candidate is discarded

**Why is this needed?** Instruction-tuned models add explanation prefixes ("Sure! Here is...") and trailing commentary, even when prompted to output only SQL. A naive `startswith("SELECT")` check fails for these outputs.

**`_StopOnSemicolon` (→ `nl2sql/core/llm.py:84`)** is a `StoppingCriteria` that halts the HuggingFace `generate()` call at the first `;` token *during* generation. This prevents the model from continuing with prose explanations before `extract_first_select` is even called — reducing inference cost and avoiding long garbled continuations.

**`_normalize_spaced_keywords` (→ `nl2sql/core/sql_guardrails.py:14`)** fixes tokenizer artefacts like `S E L E C T` (some decoders emit SQL keywords character by character). This runs in `clean_candidate_with_reason()`, the guardrails layer shown in Section 4.

**Design alternatives not taken:**
- Structured output / JSON-mode: constrains generation to a JSON schema and avoids parsing entirely. Not universally supported by local HuggingFace models, and introduces JSON-escaping errors for SQL strings.
- Extract only fenced blocks: some models omit fences — this would miss valid SQL in those outputs.

In [ ]:
# --- Case 1: typical noisy model output with prose prefix and trailing comment ---
RAW_NOISY = """Sure! Here is the SQL for your question:

SELECT c.country, COUNT(o.orderNumber) AS order_count
FROM customers c
JOIN orders o ON c.customerNumber = o.customerNumber
GROUP BY c.country ORDER BY order_count DESC LIMIT 3;

This joins the customers and orders tables and returns the top 3 countries."""

print("Case 1 — prose prefix + trailing explanation:")
print(f"  input: {repr(RAW_NOISY[:60])}...")
print(f"  extract_first_select() -> {extract_first_select(RAW_NOISY)}")
print()

# --- Case 2: markdown fences stripped before scanning ---
RAW_FENCED = "```sql\nSELECT productName FROM products;\n```"
print("Case 2 — markdown-fenced SQL:")
print(f"  input: {repr(RAW_FENCED)}")
print(f"  extract_first_select() -> {extract_first_select(RAW_FENCED)}")
print()

# --- Case 3: prose-FROM rejection (stopword after FROM) ---
RAW_PROSE = "I can answer from the results I have."
print("Case 3 — prose-FROM rejection ('the' is a stopword):")
print(f"  input: {repr(RAW_PROSE)}")
print(f"  extract_first_select() -> {extract_first_select(RAW_PROSE)!r}  (None = rejected)")
print()

# --- Case 4: no SELECT at all ---
RAW_NO_SQL = "I don't know how to answer that question."
print("Case 4 — no SELECT in output:")
print(f"  input: {repr(RAW_NO_SQL)}")
print(f"  extract_first_select() -> {extract_first_select(RAW_NO_SQL)!r}  (None = fallback to raw text)")
print()

# --- Case 5: spaced keywords from a tokenizer artefact ---
RAW_SPACED = "S E L E C T country F R O M customers;"
print("Case 5 — spaced tokenizer artefact (fixed by _normalize_spaced_keywords before extract):")
print(f"  input: {repr(RAW_SPACED)}")
fixed = _normalize_spaced_keywords(RAW_SPACED)
print(f"  after _normalize_spaced_keywords(): {repr(fixed)}")
print(f"  extract_first_select(fixed): {extract_first_select(fixed)}")

---
## 4. SQL Guardrails — `clean_candidate_with_reason()`

**What it does.** `clean_candidate_with_reason()` (→ `nl2sql/core/sql_guardrails.py:53`) is the security/safety gate applied *before* schema validation and execution. It:
1. Calls `_normalize_spaced_keywords()` to fix tokenizer artefacts
2. Strips markdown fences
3. Calls `extract_first_select()` to isolate the first SELECT statement
4. Checks for **DML tokens** (`INSERT`, `UPDATE`, `DELETE`, `DROP`, `ALTER`, `TRUNCATE`, `CREATE`, `GRANT`, `REVOKE`) — rejects with `"forbidden_sql"` if found

**Returns** `(sql, "ok")` on success or `(None, reason_code)` on rejection.

**Why a structured reason code?** The reason code feeds directly into the ReAct repair prompt (Section 7). `"forbidden_sql"` tells the repair model *what went wrong*, enabling targeted correction rather than blind regeneration.

**Why block DML?** The pipeline runs against a live MySQL database. An unrestricted generation could produce `DROP TABLE customers;` — structurally valid SQL that would destroy data. DML blocking is a hard safety constraint.

**Design alternatives not taken:**
- Read-only DB user: a valid defence-in-depth measure but does not give the *model* actionable feedback when it generates DML.
- Allowlist of SQL verbs: identical to the DML blocklist approach; the blocklist is slightly more conservative (safer to enumerate what to block than what to allow).

In [ ]:
# --- Demo: every rejection path in clean_candidate_with_reason() ---

guardrail_cases = [
    ("empty input",            ""),
    ("no SELECT",              "Just count the rows from orders."),
    ("DML — DROP",             "DROP TABLE customers;"),
    ("DML — UPDATE",           "UPDATE customers SET creditLimit = 0;"),
    ("good SQL",               "SELECT c.country, COUNT(*) FROM customers c JOIN orders o ON c.customerNumber = o.customerNumber GROUP BY c.country;"),
    ("fenced good SQL",        "```sql\nSELECT productName FROM products;\n```"),
]

print(f"  {'input (truncated)':<40}  {'result sql':<40}  reason")
print(f"  {'-'*40}  {'-'*40}  ------")
for label, raw in guardrail_cases:
    sql, reason = clean_candidate_with_reason(raw)
    sql_display = sql[:38] + "…" if sql and len(sql) > 40 else (sql or "None")
    print(f"  {label:<40}  {sql_display:<40}  {reason}")

print()
print("Structured reason codes let the repair prompt tell the model exactly what to fix.")
print("Example repair hint for 'select_star_forbidden':")
print("  'Do not use SELECT *. Return only the smallest set of columns needed.'")
print("  (see nl2sql/agent/react_pipeline.py:_repair_hint)")

---
## 5. Post-Processing — `guarded_postprocess()`

**What it does.** `guarded_postprocess()` (→ `nl2sql/core/postprocess.py:56`) applies two conservative transformations after extraction:
1. **`first_select_only()`** — trims everything after the first complete SQL statement (catches models that generate multiple statements)
2. **`_strip_order_by_limit()`** — removes `ORDER BY` and `LIMIT` *unless* `RANKING_HINT_RE` fires on the NLQ

**Why strip ORDER BY?**  The EX metric compares result sets as `Counter(pred_rows) == Counter(gold_rows)` — it is order-insensitive by design (following Spider evaluation convention [22]). A spurious `ORDER BY total DESC` causes an EX failure only when the database returns the same rows in a *different* order than the gold SQL. At k=0 the model adds `ORDER BY` to nearly every query regardless of whether the question asks for ranking; stripping it removes a systematic source of false negatives.

**Why keep ORDER BY for ranking queries?** Questions containing words like `top`, `highest`, `most`, `first`, `rank` explicitly request an ordered result — `LIMIT 3` without `ORDER BY` would produce non-deterministic output. `RANKING_HINT_RE` is a conservative allowlist.

**Design alternatives not taken:**
- Always strip ORDER BY: would fail on ranking questions where LIMIT depends on order.
- Never strip ORDER BY: produces systematic false-negative EX failures at k=0 where models add spurious ordering.

In [ ]:
print("RANKING_HINT_RE pattern:", RANKING_HINT_RE.pattern)
print()

postprocess_cases = [
    # (nlq, sql, description)
    ("What is the total amount paid by each customer?",
     "SELECT customerNumber, SUM(amount) AS total FROM payments GROUP BY customerNumber ORDER BY total DESC;",
     "no ranking keyword → ORDER BY stripped"),

    ("Top 3 countries by number of orders.",
     "SELECT c.country, COUNT(*) FROM customers c JOIN orders o ON c.customerNumber = o.customerNumber GROUP BY c.country ORDER BY COUNT(*) DESC LIMIT 3;",
     "ranking keyword 'Top' → ORDER BY + LIMIT preserved"),

    ("List all customers in France and Germany.",
     "SELECT customerName FROM customers WHERE country IN ('France','Germany') ORDER BY customerName;",
     "no ranking keyword → ORDER BY stripped despite 'and'"),
]

for nlq, sql, desc in postprocess_cases:
    result = guarded_postprocess(sql, nlq)
    ranking_match = RANKING_HINT_RE.search(nlq)
    keyword = f"'{ranking_match.group(0)}'" if ranking_match else "none"
    changed = "CHANGED" if result != sql else "unchanged"
    print(f"  NLQ:    {nlq}")
    print(f"  Ranking keyword found: {keyword}")
    print(f"  Input:  {sql}")
    print(f"  Output: {result}  [{changed}]")
    print(f"  → {desc}")
    print()

---
## 6. Pre-Execution Validation — `validate_sql()`

**What it does.** `validate_sql()` (→ `nl2sql/core/validation.py:74`) applies a four-stage pre-execution check before touching the database. The pipeline short-circuits at the first failure and returns a structured `{"valid": bool, "reason": str}` dict.

| Stage | Check | Reason code |
|-------|-------|-------------|
| 1 | Empty or whitespace-only string | `"empty_sql"` |
| 2 | `clean_candidate_with_reason()` — DML tokens, no SELECT | `"clean_reject:no_select"`, `"clean_reject:forbidden_sql"` |
| 3 | `SELECT *` guard (unless NLQ asks for all columns) | `"select_star_forbidden"` |
| 4 | `schema_validate()` — unknown table names in FROM/JOIN | `"unknown_table:<name>"` |

**Why `SELECT *` is blocked.** `SELECT *` returns every column; the gold SQL almost always names specific columns. `Counter(pred_rows) != Counter(gold_rows)` even if the data is correct, because the row tuples have different lengths. The NLQ override (`_SELECT_STAR_ALLOW_RE`) handles genuine "show all" questions.

**Why not column-level validation?** Column aliases (e.g. `COUNT(*) AS n`) and qualified references (`c.customerNumber`) cause too many false rejections. Column existence errors are caught at execution time and fed to the repair loop.

**Why not syntax checking?** SQL syntax checking (via `sqlparse`, `pyparsing`) is fragile across MySQL dialects. Runtime execution already returns syntax errors with precise messages — better to pass those to the repair prompt than invent a parallel parser.

**Design alternatives not taken:**
- Always execute and check the error: uses a DB round-trip for every candidate; pre-validation is cheaper and gives more actionable repair codes.

In [ ]:
# --- Demo: parse_schema_text builds the index validate_sql uses ---
tables, table_cols = parse_schema_text(schema_text)
print("parse_schema_text() index:")
print(f"  tables known: {sorted(tables)}")
print()

# --- Demo: all validation failure paths ---
validation_cases = [
    ("valid SQL",
     "SELECT c.country, COUNT(*) FROM customers c JOIN orders o ON c.customerNumber = o.customerNumber GROUP BY c.country;",
     NLQ),
    ("unknown table 'invoices'",
     "SELECT customerNumber FROM invoices WHERE amount > 1000;",
     NLQ),
    ("column 'country' not in orders (not caught — column checking is intentionally omitted)",
     "SELECT country, COUNT(*) FROM orders GROUP BY country;",
     NLQ),
    ("SELECT * without 'all columns' in NLQ",
     "SELECT * FROM customers;",
     "List customers."),
    ("SELECT * allowed because NLQ says 'all columns'",
     "SELECT * FROM customers;",
     "Show all columns of customers."),
    ("DML — blocked by clean_candidate_with_reason",
     "DELETE FROM customers WHERE creditLimit < 1000;",
     NLQ),
]

print(f"  {'description':<52}  result")
print(f"  {'-'*52}  ------")
for desc, sql, nlq_ctx in validation_cases:
    r = validate_sql(sql, schema_text, nlq=nlq_ctx)
    status = "PASS" if r['valid'] else f"FAIL [{r['reason']}]"
    print(f"  {desc:<52}  {status}")

---
## 7. Execution-Guided Repair — `repair_sql()` and the ReAct Loop

**What it does.** `run_react_pipeline()` (→ `nl2sql/agent/react_pipeline.py:278`) implements a Reason+Act+Observe loop (Yao et al. [19]). Each call produces a trace — a list of dicts recording every step. The loop actions are:

```
get_schema → generate_sql → [validate_sql → repair_sql]* → run_sql → stop
```

`_ReactState` (→ `react_pipeline.py:197`) carries mutable loop state: `current_sql`, `repairs_used`, `step`, `trace`. Module-level functions (`_add_trace`, `_stop_with`, `_try_repair`) operate on this state object rather than using `nonlocal` closures — this keeps each function independently testable.

**Why `max_repairs=2`?** The repair budget allows both a *validation repair* (fixing a schema error before execution) and an *execution repair* (fixing a runtime SQL error). A budget of 1 would mean an execution repair can never fire if validation already repaired once — wasting the most informative signal (the actual DB error).

**Why zero-shot repair?** `repair_sql()` uses a zero-shot prompt (schema + bad SQL + error message). NLQ→SQL exemplars from the generation pool are the *wrong format* for the repair task — the repair task is `(bad_sql, error) → fix`, not `(nlq) → sql`. Including generation exemplars in the repair prompt causes task confusion in small models (DIN-SQL [5]). A corpus of `(broken_sql, error, corrected_sql)` triples would be needed for few-shot repair, which is not available here.

**Why a separate `SQL_REPAIR_SYSTEM_PROMPT`?** The generator system prompt says "Write one SQL SELECT query"; the repair prompt says "Return one corrected SQL SELECT" given an error. These are different tasks requiring different role framing. Reusing the generator prompt for repair causes models to ignore the error context.

In [ ]:
# --- Simulate a ReAct trace (no model needed) ---
# This shows the exact structure that run_react_pipeline() returns.
# In a real run, generate_sql() and repair_sql() produce the SQL strings.

import json

NLQ_REACT = "Top 3 countries by number of orders."

# Step 1: get_schema
trace_step1 = {
    "step": 1, "action": "get_schema",
    "observation": {"schema_lines": len(schema_text.splitlines()), "tables": sorted(tables)},
    "reason": None,
}

# Step 2: generate_sql (model produces wrong table reference)
bad_sql = "SELECT country, COUNT(*) FROM orders GROUP BY country ORDER BY COUNT(*) DESC LIMIT 3;"
trace_step2 = {
    "step": 2, "action": "generate_sql",
    "observation": {"sql": bad_sql},
    "reason": None,
}

# Step 3: validate_sql → FAIL (column 'country' not in orders)
# Note: validate_sql checks TABLE names not column names — this specific SQL actually passes
# validation (country is not a table name). To show the schema-rejection path, use an
# unknown table name:
bad_sql_table = "SELECT country, COUNT(*) FROM order_log GROUP BY country ORDER BY COUNT(*) DESC LIMIT 3;"
v_result = validate_sql(bad_sql_table, schema_text, nlq=NLQ_REACT)
trace_step3 = {
    "step": 3, "action": "validate_sql",
    "observation": v_result,
    "reason": v_result.get("reason"),
}

# Step 4: repair_sql (model fixes 'order_log' → correct JOIN)
good_sql = "SELECT c.country, COUNT(*) AS n FROM customers c JOIN orders o ON c.customerNumber = o.customerNumber GROUP BY c.country ORDER BY n DESC LIMIT 3;"
trace_step4 = {
    "step": 4, "action": "repair_sql",
    "observation": {"sql": good_sql, "repairs_used": 1},
    "payload": {"error": "validate_sql:unknown_table:order_log"},
}

# Step 5: validate_sql → PASS
v_good = validate_sql(good_sql, schema_text, nlq=NLQ_REACT)
trace_step5 = {
    "step": 5, "action": "validate_sql",
    "observation": v_good,
    "reason": None,
}

# Step 6: run_sql → success
trace_step6 = {
    "step": 6, "action": "run_sql",
    "observation": {"success": True, "rowcount": 3, "error": None, "sql": good_sql},
    "reason": None,
}

# Step 7: stop
trace_step7 = {"step": 7, "action": "stop", "reason": "success", "observation": {"sql": good_sql}}

full_trace = [trace_step1, trace_step2, trace_step3, trace_step4, trace_step5, trace_step6, trace_step7]

print("Simulated run_react_pipeline() trace:")
print(f"  NLQ: {NLQ_REACT!r}")
print(f"  Final SQL: {good_sql}")
print()
for step in full_trace:
    action = step["action"]
    obs    = step.get("observation", {})
    reason = step.get("reason") or step.get("payload", {}).get("error", "")
    obs_str = json.dumps(obs)[:80]
    print(f"  step {step['step']:>2}  [{action:<14}]  obs: {obs_str}{'…' if len(json.dumps(obs)) > 80 else ''}")
    if reason:
        print(f"         reason: {reason}")
print()
print(f"  repairs_used: 1 of 2  (max_repairs=2 — one validation + one execution repair allowed)")
print()
print("Repair prompt format (zero-shot — no exemplars):")
print("  [system]  SQL_REPAIR_SYSTEM_PROMPT")
print("  [user]    Schema Details: <schema_text>")
print(f"  [user]    Natural Language Question: {NLQ_REACT}")
print(f"           Previous SQL: {bad_sql_table}")
print(f"           Observed Error: validate_sql:unknown_table:order_log")
print(f"           Return one corrected SQL SELECT.")

---
## 8. Evaluation Metrics — VA, EM, EX, TS

**What they measure and why each is needed:**

| Metric | Formula | Purpose |
|--------|---------|---------|
| **VA** | SQL executes without error | Partial credit — correct structure, wrong semantics still counts |
| **EM** | `normalize_sql(pred) == normalize_sql(gold)` | String-level correctness; too strict (alias differences fail) |
| **EX** | `Counter(pred_rows) == Counter(gold_rows)` | Semantic correctness; order-insensitive; **primary metric** |
| **TS** | EX across 3 DB variants | Tests robustness to DB-specific data (Yu et al. [21]) |

**Why EX over EM?** EM fails for semantically equivalent SQL that differs in alias, keyword case, or whitespace. Two queries that return identical data are functionally equivalent for any downstream use — EX captures this; EM does not.

**Why `Counter` for EX?** `Counter(rows)` makes comparison order-insensitive and handles duplicate rows correctly. This matches Spider's evaluation convention [22].

**Why `normalize_sql` for EM?** `normalize_sql` strips semicolons, collapses whitespace, and lowercases — removing purely cosmetic differences. EM is still strict (different alias → EM=False) but at least does not penalise formatting noise.

**Why TS?** A model that memorises specific values in the classicmodels DB would score high on EX but fail on a DB with different customer records. TS tests on 3 database variants to measure generalisation (used as a robustness corroboration, not a primary claim).

In [ ]:
gold_sql   = "SELECT c.country, COUNT(*) FROM customers c JOIN orders o ON c.customerNumber = o.customerNumber GROUP BY c.country ORDER BY COUNT(*) DESC LIMIT 3;"
pred_alias = "SELECT c.country, COUNT(*) AS n FROM customers c JOIN orders o ON c.customerNumber = o.customerNumber GROUP BY c.country ORDER BY n DESC LIMIT 3;"
pred_wrong = "SELECT country, COUNT(*) FROM orders GROUP BY country ORDER BY COUNT(*) DESC LIMIT 3;"

print("normalize_sql() — strips semicolons, lowercases, collapses whitespace:")
print(f"  gold:       {normalize_sql(gold_sql)[:80]}…")
print(f"  alias 'n':  {normalize_sql(pred_alias)[:80]}…")
print()

print("EM comparison (normalize_sql):")
print(f"  gold vs gold:            EM = {normalize_sql(gold_sql) == normalize_sql(gold_sql)}")
print(f"  gold vs alias SQL:       EM = {normalize_sql(gold_sql) == normalize_sql(pred_alias)}  ← different alias 'n'")
print(f"  gold vs wrong table SQL: EM = {normalize_sql(gold_sql) == normalize_sql(pred_wrong)}")
print()

from collections import Counter
gold_rows    = [("France", 10), ("USA", 8), ("Germany", 6)]
pred_reorder = [("USA", 8), ("Germany", 6), ("France", 10)]
pred_wrong_r = [("France", 10), ("USA", 8), ("Spain", 5)]

print("EX comparison (Counter — order-insensitive):")
for label, pred in [("same order", gold_rows), ("reordered", pred_reorder), ("wrong value", pred_wrong_r)]:
    ex = Counter(pred) == Counter(gold_rows)
    print(f"  {label:<15}  EX = {ex}  pred={pred}")
print()
print("Key insight: EX=True for reordered rows (same data, different ORDER BY).")
print("This is why guarded_postprocess strips spurious ORDER BY before evaluation.")

---
## 9. Results Across All Conditions

All 8 primary conditions × 3 seeds (n=200 per seed → n=600 paired items per condition). Results from `results/analysis/stats_mean_median_std.csv`.

**Core finding:** Few-shot Base k=3 beats QLoRA k=3 on both models. Fine-tuning did not improve over ICL in this constrained (single-GPU, ~24GB VRAM) setting.

In [ ]:
# all 8 conditions — means from results/analysis/stats_mean_median_std.csv
conditions = [
    ("Llama Base  k=0", 0.765, 0.005, 0.410, None),
    ("Llama Base  k=3", 0.870, 0.298, 0.517, 0.559),
    ("Llama QLoRA k=0", 0.815, 0.000, 0.440, None),
    ("Llama QLoRA k=3", 0.855, 0.285, 0.465, None),
    ("Qwen  Base  k=0", 0.805, 0.010, 0.480, None),
    ("Qwen  Base  k=3", 0.892, 0.357, 0.610, 0.667),
    ("Qwen  QLoRA k=0", 0.805, 0.010, 0.435, None),
    ("Qwen  QLoRA k=3", 0.895, 0.302, 0.530, 0.566),
]

print(f"  {'Condition':<20}  {'VA':>6}  {'EM':>6}  {'EX':>6}  {'TS':>6}")
print(f"  {'-'*20}  {'------':>6}  {'------':>6}  {'------':>6}  {'------':>6}")
for label, va, em, ex, ts in conditions:
    ts_str = f"{ts:.3f}" if ts else "  N/A"
    flag   = "  <--" if "k=3" in label else ""
    print(f"  {label:<20}  {va:>6.3f}  {em:>6.3f}  {ex:>6.3f}  {ts_str:>6}{flag}")

---
## 10. Statistical Analysis

**Test choice rationale:**
- **Wilcoxon signed-rank** (primary): binary metrics (EX ∈ {0,1}) are non-normally distributed — all Shapiro-Wilk tests reject normality. Wilcoxon requires only ordinal data and handles ties correctly (`zero_method='wilcox'` discards tied pairs).
- **Paired t-test** (corroboration): justified by CLT at n=600; used to cross-check Wilcoxon decisions. Both tests agree on all 12 EX comparisons.
- **BH-FDR correction**: 12 correlated comparisons per metric family — Benjamini-Hochberg controls the false discovery rate at 5%.
- **Cohen's d_z** (effect size): `mean_diff / std(paired_differences)` — the paired variant appropriate for within-subject designs.

**Key significance findings:**
- Llama QLoRA k0→k3 EX gain: **non-significant** (Wilcoxon p=0.055, BH=0.055) — fine-tuning suppressed ICL benefit in Llama.
- All other k0→k3 EX comparisons: **significant** (p < 0.001).
- Base vs QLoRA at k=3: **significant** on both models — core dissertation claim.

In [ ]:
# paired t-test summary — from results/analysis/stats_paired_ttests.csv
tests = [
    ("Llama Base k0->k3",     "EX", 0.410, 0.517, 1.10e-11, True),
    ("Llama QLoRA k0->k3",    "EX", 0.440, 0.465, 0.0547,   False),   # NOT significant
    ("Qwen Base k0->k3",      "EX", 0.490, 0.610, 8.21e-09, True),
    ("Qwen QLoRA k0->k3",     "EX", 0.435, 0.530, 1.37e-09, True),
    ("Llama Base vs QLoRA k3","EX", 0.517, 0.465, 7.40e-04, True),    # core claim
    ("Qwen Base vs QLoRA k3", "EX", 0.610, 0.530, 8.60e-06, True),    # core claim
]

print(f"  {'comparison':<26}  {'M':<3}  {'left':>6}  {'right':>6}  {'delta':>7}  {'p':>10}  sig?")
print(f"  {'-'*26}  ---  {'------':>6}  {'------':>6}  {'-------':>7}  {'----------':>10}  ----")
for comp, m, lv, rv, p, sig in tests:
    delta = (rv - lv) * 100
    p_str = f"{p:.2e}" if p < 0.001 else f"{p:.4f}"
    note  = "yes" if sig else "NO (p=0.055)"
    print(f"  {comp:<26}  {m:<3}  {lv:>6.3f}  {rv:>6.3f}  {delta:>+6.1f}pp  {p_str:>10}  {note}")

print()
print("Llama QLoRA k0->k3 EX gain is NOT significant — fine-tuning may suppress ICL benefit")
print()

# ReAct (3 seeds: 7, 17, 27)
print("ReAct mean vs Llama Base k=3  (3 seeds, n=600 matched items):")
for m, bv, rv in zip(["VA","EM","EX","TS"], [0.870,0.298,0.517,0.559], [0.925,0.458,0.628,0.661]):
    print(f"  {m}: {bv:.3f} -> {rv:.3f}  ({(rv-bv)*100:+.1f}pp)")
print("  [ReAct means computed from seeds 7, 17, 27]")

---
## 11. Summary — Pipeline and Findings

### Pipeline recap
| Stage | Function | Location |
|-------|----------|----------|
| Schema encoding | `schema_to_text()` | `agent/agent_tools.py:52` |
| Prompt assembly | `make_few_shot_messages()` | `core/prompting.py:23` |
| Generation | `generate_sql_from_messages()` + `_StopOnSemicolon` | `core/llm.py:67` |
| SQL extraction | `extract_first_select()` | `core/llm.py:38` |
| Guardrails | `clean_candidate_with_reason()` | `core/sql_guardrails.py:53` |
| Post-processing | `guarded_postprocess()` | `core/postprocess.py:56` |
| Validation | `validate_sql()` + `schema_validate()` | `core/validation.py:74` |
| Repair loop | `run_react_pipeline()` + `repair_sql()` | `agent/react_pipeline.py:278` |
| Evaluation | `execution_accuracy()` + `normalize_sql()` | `evaluation/eval.py` |

### Core empirical findings (EX metric, n=600 per condition)
- **k=0 → k=3 uplift:** +10.7pp (Llama) / +12.0pp (Qwen) — ICL is the dominant driver of accuracy.
- **Base k=3 beats QLoRA k=3:** +5.2pp (Llama, p<0.001) / +8.0pp (Qwen, p<0.001) — fine-tuning does not help in the constrained setting.
- **Llama QLoRA k0→k3:** non-significant (p=0.055) — fine-tuning suppressed ICL benefit.
- **ReAct adds:** +11.2pp (Llama) / +4.8pp (Qwen) over k=3 baselines via execution-guided repair.

> **Headline:** Few-shot ICL (k=3) with a frozen model outperforms QLoRA fine-tuning in a single-GPU, resource-constrained setting. ReAct extends this further at the cost of multi-step inference.